# Lesson 10 - Wakala wa AI katika Uzalishaji

Katika somo hili utajifunza **mifumo ya uzalishaji** kwa wakala wa AI ukitumia Microsoft Agent Framework (Python). Tunashughulikia:

- **Uchunguzi** — kuongeza upimaji wa wakati na uandishi wa kumbukumbu kwa mwingiliano wa wakala
- **Tathmini** — kutumia wakala mtathmini kuhesabu ubora wa majibu
- **Usimamizi wa gharama** — mikakati ya ufanisi wa tokeni na uteuzi wa modeli

Senaria ni **wakala wa usafiri** anayesaidia watumiaji kupanga safari, kwa ufuatiliaji na tathmini zilizowekwa juu.


## Mipangilio


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mawazo ya Uzalishaji

Kuhamisha mawakala wa AI kutoka kwa prototypes hadi uzalishaji kunahitaji umakini wa makini kwa nguzo tatu:

1. **Uwezo wa Kuchunguza** — Unahitaji uonekano wa kile wakala anachofanya, ni muda gani huchukua, na ni zana gani anazitumia. Bila kufuatilia na kurekodi, kutatua matatizo ya uzalishaji ni karibu haiwezekani.

2. **Tathmini** — Ukaguzi wa ubora wa kiotomatiki huhakikisha majibu ya wakala yanabaki kuwa sahihi, kamili, na ya msaada kwa muda. Wakala wa mtathmini anaweza kutoa alama kwa majibu kulingana na vigezo vilivyowekwa.

3. **Usimamizi wa Gharama** — Matumizi ya tokeni huathiri gharama moja kwa moja. Mikakati kama ubora wa maelekezo, uchaguzi wa mfano, na uhifadhi husaidia kudhibiti matumizi bila kuathiri ubora.


## Kujenga Wakala Anayeonekana

Tunaelezea zana za usafiri na kufunga wito wa wakala na kipima muda ili tuweze kufuatilia ucheleweshaji. Katika uzalishaji ungewaiunganisha na OpenTelemetry au mfumo mwingine wa ufuatiliaji.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mifumo ya Tathmini

Mfumo wa kawaida wa uzalishaji ni kutumia wakala wa pili kama **mtathmini**. Mtathmini hutoa alama kwa jibu la wakala mkuu dhidi ya vigezo vilivyoainishwa kama ukamilifu, usahihi, na msaada.

Hii inaruhusu:
- Milango ya ubora ya kiotomatiki kabla ya majibu kufikia watumiaji
- Ugunduzi wa kurudi nyuma wakati maelekezo au mifano inabadilika
- Ufuatiliaji endelevu wa utendaji wa wakala kwa wakati


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mikakati ya Usimamizi wa Gharama

Kudhibiti gharama ni muhimu kwa maajenti wa AI wa uzalishaji. Hapa kuna mikakati muhimu:

| Mkakati | Maelezo |
|---|---|
| **Uboreshaji wa maagizo** | Weka maelekezo ya mfumo kuwa mafupi. Ondoa muktadha uliorudiwa ili kupunguza tokeni za ingizo. |
| **Uchaguzi wa mfano** | Tumia mifano midogo, nafuu (km. GPT-4o-mini) kwa kazi rahisi kama vile upangaji au uondoaji, na uhifadhi mifano mikubwa kwa hoja ngumu. |
| **Kuweka hifadhidata** | Weka matokeo ya zana na maswali ya mara kwa mara ili kuepuka miito ya API isiyohitajika. |
| **Mikakati ya tokeni** | Weka vizingiti vya `max_tokens` ili kuzuia majibu marefu yasiyotarajiwa. |
| **Kukusanya** | Kusanya maswali mengi ya mtumiaji na kuyatumia kwa miito moja ya API pale inapowezekana. |

Kwa vitendo, mbinu ya ngazi nyingi hufanya kazi vizuri: elekeza maombi rahisi kwa mfano wa haraka, nafuu na ongeza tu maswali magumu kwa mfano wenye uwezo zaidi.


## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kuongeza ufanikishaji** kwa mwingiliano wa wakala kwa kutumia upimaji wa wakati na kurekodi, kuweka msingi wa ufuatiliaji na uangalizi.
2. **Kutathmini majibu ya wakala** kiotomatiki kwa kutumia wakala mtathmini anayepima ukamilifu, usahihi, na msaada.
3. **Kudhibiti gharama** kupitia uboreshaji wa maagizo, uchaguzi wa mfano, uhifadhi, na bajeti za tokeni.

Mifumo hii ya uzalishaji husaidia kuhakikisha kuwa mawakala wa AI wako wa kuaminika, yanayopimika, na yenye gharama nafuu kwa kiwango kikubwa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
